In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
events = spark.table("day8_catalog.ecommerce.events_silver")

events.printSchema()
events.count()


In [0]:
events.describe("price").show()


In [0]:
events.groupBy("event_type") \
      .agg(
          F.count("*").alias("count"),
          F.round(F.avg("price"), 2).alias("avg_price"),
          F.round(F.stddev("price"), 2).alias("std_price")
      ).show()


In [0]:
events_time = events.withColumn(
    "event_date", F.to_date("event_time")
).withColumn(
    "is_weekend",
    F.dayofweek("event_date").isin([1, 7])  # Sunday & Saturday
)


In [0]:
events_time.groupBy("is_weekend", "event_type") \
    .count() \
    .orderBy("is_weekend", "event_type") \
    .show()


In [0]:
conversion_by_day = (
    events_time
    .groupBy("is_weekend", "event_type")
    .count()
    .groupBy("is_weekend")
    .pivot("event_type")
    .sum("count")
    .withColumn(
        "conversion_rate",
        F.round(F.col("purchase") * 100.0 / F.col("view"), 2)
    )
)

conversion_by_day.show()


In [0]:
events_corr = events.withColumn(
    "is_purchase", F.when(F.col("event_type") == "purchase", 1).otherwise(0)
)

events_corr.stat.corr("price", "is_purchase")


In [0]:
window_user = Window.partitionBy("user_id").orderBy("event_time")

features_df = (
    events
    .withColumn("hour", F.hour("event_time"))
    .withColumn("day_of_week", F.dayofweek("event_time"))
    .withColumn("price_log", F.log(F.col("price") + 1))
    .withColumn(
        "time_since_first_event",
        F.unix_timestamp("event_time") -
        F.unix_timestamp(F.first("event_time").over(window_user))
    )
)


In [0]:
features_df.select(
    "user_id",
    "event_time",
    "hour",
    "day_of_week",
    "price",
    "price_log",
    "time_since_first_event"
).show(10)


In [0]:
features_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("day8_catalog.ecommerce.ml_features_day11")


In [0]:
spark.table("day8_catalog.ecommerce.ml_features_day11").count()
